# Decision Trees from First Principles

## Mathematical Theory

**Entropy**: $H(S) = -\sum_{i=1}^{c} p_i \log_2 p_i$

**Information Gain**: $IG(S, A) = H(S) - \sum_{v} \frac{|S_v|}{|S|} H(S_v)$

**Gini Impurity**: $Gini(S) = 1 - \sum_{i=1}^{c} p_i^2$

In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, \
                            f1_score, confusion_matrix, roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split

In [40]:
SEED = 42
np.random.seed(SEED)

In [41]:
# Load and prepare data
df = pd.read_csv('zuu crew scores.csv')
df = df[df['CourseName'] == 'Foundations of ML'].drop(columns=['CourseName', 'MemberName'])

threshold = df['CapstoneScore'].median()
df['CapstoneScore'] = (df['CapstoneScore'] >= threshold).map({True: 'Pass', False: 'Fail'})

X = df.drop(columns=['CapstoneScore']).astype('float32')
y = df['CapstoneScore']

X_train, X_test, y_train, y_test = train_test_split(
                                                    X, y,
                                                    test_size=0.2, 
                                                    random_state=SEED, 
                                                    stratify=y
                                                    )
print(f"Data prepared: {df.shape}")

Data prepared: (72, 8)


In [42]:
Counter(y_train).most_common(1)[0][0]

'Pass'

In [43]:
# Decision Tree from Scratch
class TreeNode:
    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.left = None
        self.right = None
        self.prediction = None

class DecisionTree:
    def __init__(self, maximum_depth=5, criterion='gini'):
        self.maximum_depth = maximum_depth
        self.criterion = criterion
        self.tree = None
    
    def _impurity(self, y):
        if len(y) == 0:
            return 0
        counts = Counter(y)
        n = len(y)
        
        if self.criterion == 'gini':
            return 1 - sum((count/n)**2 for count in counts.values())
        else:  # entropy
            return -sum((count/n) * np.log2(count/n) for count in counts.values() if count > 0)
    
    def _best_split(self, X, y):
        best_gain, best_feature, best_threshold = 0, None, None
        current_impurity = self._impurity(y)
        
        for feature_idx in range(X.shape[1]):
            values = X.iloc[:, feature_idx].unique()
            for val in values:
                left_mask = X.iloc[:, feature_idx] <= val
                if left_mask.sum() == 0 or left_mask.sum() == len(y):
                    continue
                
                y_left, y_right = y[left_mask], y[~left_mask]
                gain = current_impurity - (len(y_left)/len(y) * self._impurity(y_left) + 
                                          len(y_right)/len(y) * self._impurity(y_right))
                
                if gain > best_gain:
                    best_gain, best_feature, best_threshold = gain, feature_idx, val
        
        return best_feature, best_threshold
    
    def _build_tree(self, X, y, depth=0):
        node = TreeNode()
        node.prediction = Counter(y).most_common(1)[0][0]
        
        if depth >= self.maximum_depth or len(set(y)) == 1:
            return node
        
        feature_idx, threshold = self._best_split(X, y)
        if feature_idx is None:
            return node
        
        node.feature_index, node.threshold = feature_idx, threshold
        left_mask = X.iloc[:, feature_idx] <= threshold
        
        node.left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        node.right = self._build_tree(X[~left_mask], y[~left_mask], depth + 1)
        
        return node
    
    def fit(self, X, y):
        self.tree = self._build_tree(X, y)
        return self
    
    def _predict_sample(self, sample, node):
        if node.feature_index is None:
            return node.prediction
        
        if sample[node.feature_index] <= node.threshold:
            return self._predict_sample(sample, node.left)
        else:
            return self._predict_sample(sample, node.right)
    
    def predict(self, X):
        return np.array([self._predict_sample(X.iloc[i], self.tree) for i in range(len(X))])

In [44]:
zuu_tree_shallow = DecisionTree(maximum_depth=1)
zuu_tree_shallow.fit(X_train, y_train)

zuu_tree_deep = DecisionTree(maximum_depth=12)
zuu_tree_deep.fit(X_train, y_train)

P_shallow = zuu_tree_shallow.predict(X_test)
P_deep = zuu_tree_deep.predict(X_test)

/var/folders/b5/fr4bwywx1hl34z7s66ljm40r0000gn/T/ipykernel_77662/1835303592.py:74: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if sample[node.feature_index] <= node.threshold:


In [45]:
P_shallow, P_deep

(array(['Pass', 'Pass', 'Pass', 'Pass', 'Fail', 'Fail', 'Fail', 'Fail',
        'Pass', 'Pass', 'Fail', 'Fail', 'Pass', 'Fail', 'Fail'],
       dtype='<U4'),
 array(['Pass', 'Pass', 'Pass', 'Pass', 'Fail', 'Fail', 'Fail', 'Fail',
        'Pass', 'Pass', 'Fail', 'Fail', 'Pass', 'Fail', 'Fail'],
       dtype='<U4'))

In [46]:
# Calculate metrics
metrics = {
    'Model': ['Shallow Tree', 'Deep Tree'],
    'Precision': [
        precision_score(y_test, P_shallow, pos_label='Pass'),
        precision_score(y_test, P_deep, pos_label='Pass')
    ],
    'Recall': [
        recall_score(y_test, P_shallow, pos_label='Pass'),
        recall_score(y_test, P_deep, pos_label='Pass')
    ],
    'F1 Score': [
        f1_score(y_test, P_shallow, pos_label='Pass'),
        f1_score(y_test, P_deep, pos_label='Pass')
    ],
    'AUC': [
        roc_auc_score(y_test == 'Pass', P_shallow == 'Pass'),
        roc_auc_score(y_test == 'Pass', P_deep == 'Pass')
    ]
}

# Create and display DataFrame
pd.DataFrame(metrics).round(3)

,Model,Precision,Recall,F1 Score,AUC
0,Shallow Tree,0.714,0.714,0.714,0.732
1,Deep Tree,0.714,0.714,0.714,0.732


Homework </br>
        1. Compute Confusion Matrix</br>
        2. Visualize ROC Curve